# 03 · Experimentos MLP

Tres corridas (PPG, ECG, PPG+ECG) con **el mismo presupuesto**: AdamW(lr=1e-3, wd=1e-2), warmup de 1 época + coseno por lote,
batch 128, hasta 30 épocas, early stopping (paciencia 5) sobre `val_loss`, `clip_grad_norm_(1.0)`, semilla 0 antes de construir cada modelo.
Se registra en MLflow (`mlflow.db`). Arquitectura: `Linear(bias=False) → BatchNorm1d → ReLU → [Dropout]` ×3 (512→256→128) → `Linear(128, 2)`; He en capas ocultas y la última capa en 0. La variante base no usa Dropout; como sobreajusta en las primeras épocas (la `train_loss` baja y la `val_loss` sube) se añade la variante con `Dropout(0.2)` del plan y se elige por validación.
**Solo train y val**; test se evalúa una vez en el notebook 06.

In [1]:
import sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "bp").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, torch
print("ROOT:", ROOT, "| cuda:", torch.cuda.is_available())

ROOT: C:\Users\USUARIO\Documents\Source\dl_bp_mimic_proyecto_integrador | cuda: True


In [2]:
from bp.experiment import get_data, load_label_stats, save_val_result
from bp.train import fit
from bp.dataset import INPUT_SETS
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
y_mean, y_std = load_label_stats()
runs = {}

In [3]:
variants = [('', {}), ('_do0.2', {'dropout': 0.2})]   # (sufijo, hiperparámetros del modelo); se elige por val_loss en el notebook 06
for suffix, model_kw in variants:
    for name, inputs in INPUT_SETS.items():
        tr, va = get_data('train', inputs), get_data('val', inputs)
        res = fit('build_mlp', inputs, tr, va, y_mean, y_std, run_name='mlp' + suffix + '_' + name, model_kw=model_kw)
        save_val_result(res); runs[suffix + name] = res
        print(f"==> mlp{suffix} {name}: mejor época {res['best_epoch']} val_loss {res['best_val_loss']:.4f} "
              f"MAE val PAS/PAD {res['val_mae_pas']:.2f}/{res['val_mae_pad']:.2f} mmHg ({res['seconds']:.0f}s)")
        del tr, va; torch.cuda.empty_cache()

2026/11/04 01:30:05 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at C:\Users\USUARIO\.conda\envs\dl_bp\Lib\site-packages\mlflow\assistant\skills\mlflow-trace-instrumentation\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[mlp_ppg] ep 00 train 0.9861 val 0.9162 MAE 14.14/9.25 gn 1.06


[mlp_ppg] ep 01 train 0.9679 val 0.9103 MAE 14.16/9.23 gn 1.02


[mlp_ppg] ep 02 train 0.9502 val 0.9082 MAE 14.15/9.22 gn 0.97


[mlp_ppg] ep 03 train 0.9359 val 0.9072 MAE 14.08/9.14 gn 0.89


[mlp_ppg] ep 04 train 0.9173 val 0.9169 MAE 14.08/9.23 gn 0.84


... (43 outputs left out)

[mlp_do0.2_ppg+ecg] ep 03 train 0.8374 val 0.9530 MAE 14.37/9.41 gn 1.05


[mlp_do0.2_ppg+ecg] ep 04 train 0.7699 val 0.9911 MAE 14.47/9.68 gn 1.04


[mlp_do0.2_ppg+ecg] ep 05 train 0.6957 val 1.0293 MAE 14.79/9.70 gn 1.01


==> mlp_do0.2 ppg+ecg: mejor época 0 val_loss 0.9157 MAE val PAS/PAD 14.10/9.28 mmHg (9s)


In [4]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for name, res in runs.items():
    h = res['history']
    ax[0].plot([r['train_loss'] for r in h], label=name); ax[1].plot([r['val_loss'] for r in h], label=name)
    ax[2].plot([r['val_mae_pas_mmHg'] for r in h], label=name)
for a, t in zip(ax, ['train_loss (MSE estandarizado)', 'val_loss', 'val MAE PAS (mmHg)']):
    a.set_title(t); a.set_xlabel('época'); a.legend()
plt.tight_layout(); plt.savefig(ROOT / 'artifacts' / 'fig_curvas_mlp.png', dpi=110)
import pandas as pd
pd.DataFrame([{'entrada': k, 'mejor_epoca': r['best_epoch'], 'val_loss': r['best_val_loss'], 'MAE_PAS': r['val_mae_pas'], 'MAE_PAD': r['val_mae_pad']}
              for k, r in runs.items()])

,entrada,mejor_epoca,val_loss,MAE_PAS,MAE_PAD
0,ppg,3,0.907214,14.080805,9.144136
1,ecg,0,0.941185,14.173152,9.496849
2,ppg+ecg,0,0.921181,14.147727,9.304474
3,_do0.2ppg,6,0.900521,13.985277,9.198669
4,_do0.2ecg,1,0.937054,14.236189,9.561135
5,_do0.2ppg+ecg,0,0.915688,14.102884,9.276887
